# Introduction

The VMT Reduction Mode Shift Study relies on an analysis of real-world trips reported in the 2019 and 2021 Travel Behavior Inventory (TBI) surveys of the Minneapolis/St. Paul Region. This technical memorandum describes how those trips were processed to support this study. 

That processing is completed in the accompanying notebook `Clean and Code TBI.ipynb` stored in the project github repository.  The result of that processing is the file `tbi_merged.csv` which is stored on the project Teams site.  The resulting data file contains one record for each linked trip to be analyzed for its mode shift potential, along with associated household, person and day attributes.  

Note: To convert this notebook to an HTML or other file, use Quarto.  See https://quarto.org/docs/get-started/hello/jupyter.html


# Main Data Processing

For this study, we include data from both the 2019 (wave 1) TBI and the 2021 (wave 2) TBI.  We expect future waves to be added as they become available, allowing for changes to be tracked over time.  

Each TBI is a week-long, smartphone-enabled household travel survey that combines passive tracing of participants smartphone locations with active responses to questions, particularly on mode and purpose.  Please refer to the survey documentation for details on its implementation and initial processing.  Each wave of TBI data is provided in several main files: 

- hh.csv: One record for each household surveyed. 
- person.csv: One record for each person in those households. 
- day.csv: One record for each travel day, generally 7 days per person. 
- trip.csv: One record for each unlinked trip made by those travelers. 
- location.csv: A series of point locations and timestamps for each trip from the smartphone traces. These files are not processed further, and will only be used in this study to QA/QC the routes. 
- vehicle.csv: One record for each vehicle in the household.  This will us to link VMT reductions to the emissions reductions associated with specific vehicles.  Ignore vehicle_id in the Wave 2 trip list and instead use mode_type_detailed for matching. 

The core data processing in this task involves:

1. Merging each of the the household, person, day and trip files across the survey waves and rectifying any differences in the available fields. 
2. Retaining only the fields potentially relevant to this study. 
3. "Decoding" the data by converting numerical codes into strings with a more obvious interpretation.  For example, 995 might be replaced by "Missing" such that 995 is not inadvertently included in an average. 
4. Aggregating from unlinked trips to linked trips. 
5. Determinging the mode and reference VMT. 
6. Excluding records that are out of scope or missing important data. 

The details are explicit in the data processing notebook. Several of the steps warrant additional explanation here, as well as the question of how to use the weights in application.  

# Trip Linking

When a person makes a trip by transit, they usually combine multiple modes, such as walking and riding the bus, and often have a stop to change modes while they wait for the bus.  These "change mode" stops are coded as trip ends in the TBI data.  However, if the person were to use a different mode, they would not need to make those stops and could proceed directly to their destination.  For that reason, the TBI data identifies a "linked trip" ID allowing us to identify how those individual trips combine for an overall origin to destination.  

A similar problem arises for car trips when someone stops for gas.  The stop for gas is usually on the way to another destination, and would not be necessary if the trip were not made by car.  Therefore, we added logic to link out stops for gas when they are part of a longer chain of trips.  We keep gas stops when they are the only non-home activity on a tour (home - gas - home). 

In addition to stops for gas, the data show that a small number of trips on modes other than transit are also linked.  Spot checking showed that these appear to be reasonable, such as someone reporting changing modes from what appears to be a church van to a personal vehicle.  

Once the linked trips are identified, we merge them to create one record for each linked trip.  The origin and departure time are from the first trip.  The destination and arrival time are from the last trip.  We sum the total trip duration and distance, then re-calculate the speed.  The linked trip's mode is calculated based on a priority system, which from highest to lowest priority is: 

1. Long distance passenger mode
2. School bus
3. Transit
4. Car
5. Taxi/Ridehail/Carshare
6. Bike/Scooter
7. Walk
8. Missing/Other

For example, if a linked trip involves car-transit-walk, the linked trip is coded as a transit trip because transit is the highest priority mode.  


# Mode and Reference VMT

The available modes coded in the survey are somewhat more complicated, but we code them into the 7 modes shown above for this study.  We do so using the following equivalency:

| Detailed Mode                      | Mode                        |
| ---------------------------------- | --------------------------- |
| Long distance passenger mode       | Long distance passenger mode|
| School bus                         | School bus                  |
| Ferry                              | Transit                     |
| Rail                               | Transit                     |
| Public bus                         | Transit                     |
| Other bus                          | Transit                     |
| Shuttle                            | Transit                     |
| Transit                            | Transit                     |
| Household vehicle                  | Car                         |
| Other vehicle                      | Car                         |
| Vehicle                            | Car                         |
| Taxi                               | Taxi/Ridehail/Carshare      |
| Carshare                           | Taxi/Ridehail/Carshare      |
| For-hire vehicle                   | Taxi/Ridehail/Carshare      |
| Smartphone ridehailing service     | Taxi/Ridehail/Carshare      |
| Smartphone-app ride-hailing service| Taxi/Ridehail/Carshare      |
| Bicycle or e-bicycle               | Bike/Scooter                |
| Bike-share                         | Bike/Scooter                |
| Scooter-share                      | Bike/Scooter                |
| Micromobility                      | Bike/Scooter                |
| Walk                               | Walk                        |
| Missing: Non-response              | Missing/Other               |
| Missing                            | Missing/Other               |
| Other                              | Missing/Other               |

To calculate the total VMT in the region, and therefore the potential VMT reduction, we need to be cognizant of the vehicle occupancy.  Each record in our resulting data set is a person trip, but it is not necessarily a vehicle trip.  We calculate a vehicle trip field as follows: if the mode is Car or Taxi/Ridehail/Carshare, the number of vehicle trips is `1 / num_travelers`.  If the number of travelers is missing or unknown, we assume it is 1.  If the mode is not Car or Taxi/Ridehail/Carshare, then it becomes 0 vehicle trips.  The VMT of the record is calculated as the vehicle trips times the trip distance.  

With this accounting, if three people travel together 1 mile by car, they produce 1 vehicle trip, and 1 VMT.  If all three bike instead, we would reduce VMT by 1.  If only one bikes, our analysis would show a VMT reduction of 1/3 of a mile.  While this is not entirely realistic, all three would have the same alternative routes, so would shift together in our analysis unless there is a constraint that applies to some, but not others. 



# Exclusions

The scope of this analysis is trips made on a typical 7-day week by people in households in the 19-county study area, where both ends of the trip are within that study area.  We separately report these data for a 2019 and 2021 baseline.  To align with this scope, we exclude records as follows:

| Exclusion                                    | Trips Excluded  | Percent of VMT Excluded  |
| :------------------------------------        |   :----:        |         :---:            |
| Latitude, longitude or timestamp is missing. |    1,656        |           0%             |
| Home location outside study area.            |        0        |           0%             |
| Origin or destination outside study area.    |   37,520        |        21.5%             |
| Trip on a long-distance passenger mode.      |      166        |           0%             |
| Mode is missing.                             |   15,284        |           0%             |
| Distance is less than 0.01 miles (52.8 ft).  |      374        |           0%             |
| Distance is more than 100 miles  (5280 ft).  |      243        |         3.3%             |
| Detailed mode is ferry                       |       12        |           0%             |

The exclusions are applied in the order shown--the numbers reported do not consider that more than one exclusion may apply. The VMT is calculated as described above, which explains why the VMT reduction from excluding missing mode trips is zero.  

It is notable that almost 25% of the VMT in the survey has either one end outside the region or is trips longer than 100 miles.  Assuming the trip distances are coded correctly (it would be worth checking a few if we were to do something other than exclude these trips), it highlights how important long-distance travel may be to VMT reduction efforts.  

In addition, the data do not include trucks or commercial vehicles, which also generate substantial VMT. We think it is appropriate to exclude them from this study, but it is worth keeping in mind that there is a substantial portion of VMT that we will not shift.  

# Weighting

The data include four weights: household, person, day and trip.  Please refer to the TBI documentation for details on how they are calculated.  Each is an expansion factor, which means that when it is applied the data are "expanded" to represent the full population of the study area.  For the household and person weights, this aligns with the number of households and the population of the study area.  The day and trip weights are expanded to represent average weekday (Monday through Thursday) conditions.  

For this study, we recommend using the day weight.  When applied, the data should expand to roughly the VMT over the course of a typical weekday, accounting for people who reported different numbers of days of travel.  In the future, we might consider weekend VMT, but this requires some more thought about how to adjust the weights. 
